In [138]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

# data loading

In [139]:
raw_df = pd.read_csv("../data/train_data/train_data.csv")

# EDA

In [140]:
raw_df.head(10)

,event_id,time_to_tca,mission_id,risk,max_risk_estimate,max_risk_scaling,miss_distance,relative_speed,relative_position_r,relative_position_t,...,t_sigma_rdot,c_sigma_rdot,t_sigma_tdot,c_sigma_tdot,t_sigma_ndot,c_sigma_ndot,F10,F3M,SSN,AP
0,0,1.566798,5,-10.204955,-7.834756,8.602101,14923.0,13792.0,453.8,5976.6,...,0.147350,58.272095,0.004092,0.165044,0.002987,0.386462,89.0,83.0,42.0,11.0
1,0,1.207494,5,-10.355758,-7.848937,8.956374,14544.0,13792.0,474.3,5821.2,...,0.059672,57.966413,0.003753,0.164383,0.002933,0.386393,89.0,83.0,42.0,11.0
2,0,0.952193,5,-10.345631,-7.847406,8.932195,14475.0,13792.0,474.6,5796.2,...,0.039258,57.907599,0.003576,0.164352,0.002967,0.386381,89.0,83.0,42.0,11.0
3,0,0.579669,5,-10.337809,-7.845880,8.913444,14579.0,13792.0,472.7,5838.9,...,0.022066,57.993905,0.003298,0.164309,0.002918,0.386400,89.0,83.0,40.0,14.0
4,0,0.257806,5,-10.391260,-7.852942,9.036838,14510.0,13792.0,478.7,5811.1,...,0.015075,57.946717,0.003670,0.164172,0.003220,0.386388,89.0,83.0,40.0,14.0
5,1,6.530455,5,-7.561299,-7.254301,2.746782,2392.0,3434.0,74.3,2317.1,...,0.278354,84.677411,0.006980,0.320622,0.004199,0.047385,71.0,88.0,0.0,2.0
6,1,5.561646,5,-9.315693,-7.468904,7.223137,3587.0,3434.0,99.0,3475.4,...,0.240907,63.860857,0.006402,0.264636,0.003725,0.040020,70.0,87.0,13.0,14.0
7,1,5.226504,5,-7.422508,-7.051001,2.956639,7882.0,3434.0,-50.0,-7638.3,...,0.240198,56.764910,0.005906,0.259109,0.003588,0.083247,70.0,87.0,13.0,14.0
8,1,3.570013,5,-9.248105,-7.327533,7.425994,26899.0,3434.0,-82.0,-26067.0,...,0.124802,30.242768,0.005883,0.174956,0.003408,0.058311,71.0,87.0,21.0,5.0
9,2,6.983474,2,-10.816161,-6.601713,13.293159,22902.0,14348.0,-1157.6,-6306.2,...,0.153332,39.695541,0.009370,0.269965,0.003886,0.339406,73.0,77.0,27.0,4.0


In [141]:
col_list = []
for column in raw_df.columns:
    col_list = np.append(col_list, column) 

print(col_list)

['event_id' 'time_to_tca' 'mission_id' 'risk' 'max_risk_estimate'
 'max_risk_scaling' 'miss_distance' 'relative_speed' 'relative_position_r'
 'relative_position_t' 'relative_position_n' 'relative_velocity_r'
 'relative_velocity_t' 'relative_velocity_n' 't_time_lastob_start'
 't_time_lastob_end' 't_recommended_od_span' 't_actual_od_span'
 't_obs_available' 't_obs_used' 't_residuals_accepted' 't_weighted_rms'
 't_rcs_estimate' 't_cd_area_over_mass' 't_cr_area_over_mass' 't_sedr'
 't_j2k_sma' 't_j2k_ecc' 't_j2k_inc' 't_ct_r' 't_cn_r' 't_cn_t'
 't_crdot_r' 't_crdot_t' 't_crdot_n' 't_ctdot_r' 't_ctdot_t' 't_ctdot_n'
 't_ctdot_rdot' 't_cndot_r' 't_cndot_t' 't_cndot_n' 't_cndot_rdot'
 't_cndot_tdot' 'c_object_type' 'c_time_lastob_start' 'c_time_lastob_end'
 'c_recommended_od_span' 'c_actual_od_span' 'c_obs_available' 'c_obs_used'
 'c_residuals_accepted' 'c_weighted_rms' 'c_rcs_estimate'
 'c_cd_area_over_mass' 'c_cr_area_over_mass' 'c_sedr' 'c_j2k_sma'
 'c_j2k_ecc' 'c_j2k_inc' 'c_ct_r' 'c_cn_r

looking at the data  
A single event_id might have multiple rows leading up to the exact moment they cross paths.

In [142]:
single_event = raw_df[df["event_id"] == 0]
display(single_event[["event_id","time_to_tca","miss_distance","risk"]])

,event_id,time_to_tca,miss_distance,risk
0,0,1.566798,14923.0,-10.204955
1,0,1.207494,14544.0,-10.355758
2,0,0.952193,14475.0,-10.345631
3,0,0.579669,14579.0,-10.337809
4,0,0.257806,14510.0,-10.391260


# feature engineering

In [143]:
# extracting the target feature 

df_sorted = raw_df.sort_values(by=['event_id', 'time_to_tca'], ascending=[True, False])

final_cdms = df_sorted.groupby('event_id').last().reset_index()

# creating the target labels
targets = final_cdms[['event_id', 'risk']].copy()
targets['is_hazard'] = (targets['risk'] >= -4.0).astype(int)

# Filter for all messages issued at least 2 days before the collision
df_early = df_sorted[df_sorted['time_to_tca'] >= 2.0]

features = df_early.groupby('event_id').last().reset_index()

master_df = pd.merge(features, targets[['event_id', 'is_hazard']], on='event_id')

In [144]:
display(master_df['is_hazard'].value_counts())

is_hazard
0    11931
1       11
Name: count, dtype: int64

thats an insane imbalance :)

after the sorting we must choose the columns we want in our data set and drop the rest to reduce the noise  
next is feature selection

In [145]:
# the model cant understand coordinate features like the position, velocity vectors and angles so...
# we will turn the vectors into distance and speed but "miss_distance", "relative_speed" already exist so the vectors will just get dropped

clean_df = master_df.drop(['relative_position_r',
                            'relative_position_t', 'relative_position_n', 'relative_velocity_r',
                            'relative_velocity_t', 'relative_velocity_n'], axis=1)

In [146]:
# the angles 'geocentric_latitude', 'azimuth', 'elevation' and 'x_j2k_inc' are turned into sin and cos

# clean_df[['geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad']] = np.radians(clean_df[['geocentric_latitude', 'azimuth', 'elevation', 't_j2k_inc', 'c_j2k_inc']])
# clean_df[['geocentric_latitude_sin', 'azimuth_sin', 'elevation_sin', 't_j2k_inc_sin', 'c_j2k_inc_sin']] = np.sin(clean_df[['geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad']])
# clean_df[['geocentric_latitude_cos', 'azimuth_cos', 'elevation_cos', 't_j2k_inc_cos', 'c_j2k_inc_cos']] = np.cos(clean_df[['geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad']])

# clean_df = clean_df.drop(['geocentric_latitude', 'azimuth', 'elevation', 't_j2k_inc', 'c_j2k_inc', 'geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad'], axis=1)

angles = ['geocentric_latitude', 'azimuth', 'elevation', 't_j2k_inc', 'c_j2k_inc']
for col in angles:
    clean_df[f'{col}_sin'] = np.sin(np.radians(clean_df[col]))
    clean_df[f'{col}_cos'] = np.cos(np.radians(clean_df[col]))
    clean_df = clean_df.drop(columns=[col])

In [147]:
# now looking at the data set most of the features left are noise and must be dropped 

# after sorting and extracting the target these features will cause data leakage if not dropped
leakage_features = ['event_id', 'mission_id', 'risk', 'max_risk_estimate', 'max_risk_scaling']
clean_df = clean_df.drop(leakage_features, axis= 1)      

# the model doesn't care about the probabilities of the radar so all the radar meta data will be dropped
meta_cols = [col for col in clean_df.columns if 'lastob' in col or 'obs_' in col or 'residuals' in col or 'od_span' in col or 'rms' in col]
clean_df = clean_df.drop(columns= meta_cols)

# now dropping all the cross sectional correlations 
cross_terms = [col for col in clean_df.columns if (col.startswith('t_c') or col.startswith('c_c')) and 'type' not in col and 'span' not in col and 'covariance' not in col]
clean_df = clean_df.drop(columns=cross_terms, errors='ignore')

# dropping the left un needed columns
clean_df = clean_df.drop(columns= ["time_to_tca"])

In [148]:
col_list = []
for column in clean_df.columns:
    col_list = np.append(col_list, column) 

print(col_list)

['miss_distance' 'relative_speed' 't_rcs_estimate' 't_sedr' 't_j2k_sma'
 't_j2k_ecc' 'c_object_type' 'c_rcs_estimate' 'c_sedr' 'c_j2k_sma'
 'c_j2k_ecc' 't_span' 'c_span' 't_h_apo' 't_h_per' 'c_h_apo' 'c_h_per'
 'mahalanobis_distance' 't_position_covariance_det'
 'c_position_covariance_det' 't_sigma_r' 'c_sigma_r' 't_sigma_t'
 'c_sigma_t' 't_sigma_n' 'c_sigma_n' 't_sigma_rdot' 'c_sigma_rdot'
 't_sigma_tdot' 'c_sigma_tdot' 't_sigma_ndot' 'c_sigma_ndot' 'F10' 'F3M'
 'SSN' 'AP' 'is_hazard' 'geocentric_latitude_sin'
 'geocentric_latitude_cos' 'azimuth_sin' 'azimuth_cos' 'elevation_sin'
 'elevation_cos' 't_j2k_inc_sin' 't_j2k_inc_cos' 'c_j2k_inc_sin'
 'c_j2k_inc_cos']


In [149]:
clean_df.head()

,miss_distance,relative_speed,t_rcs_estimate,t_sedr,t_j2k_sma,t_j2k_ecc,c_object_type,c_rcs_estimate,c_sedr,c_j2k_sma,...,geocentric_latitude_sin,geocentric_latitude_cos,azimuth_sin,azimuth_cos,elevation_sin,elevation_cos,t_j2k_inc_sin,t_j2k_inc_cos,c_j2k_inc_sin,c_j2k_inc_cos
0,26899.0,3434.0,0.4030,0.000031,7001.561205,0.001028,DEBRIS,0.0045,0.008730,6880.588349,...,-0.810142,0.586234,-0.969054,0.246849,-0.016536,0.999863,0.990826,-0.135145,0.991288,0.131711
1,18763.0,14347.0,3.4479,0.000013,7158.408291,0.000863,UNKNOWN,NaN,0.000693,7168.395618,...,0.898052,0.439889,-0.275803,0.961214,-0.001004,0.999999,0.988956,-0.148208,0.938001,0.346633
2,23900.0,13574.0,0.4268,0.000019,7083.606025,0.002115,DEBRIS,0.0046,0.000487,7070.079861,...,-0.938993,0.343935,0.421301,0.906921,0.002925,0.999996,0.989897,-0.141789,0.944791,0.327675
3,33593.0,12093.0,NaN,0.000029,7082.429604,0.003942,DEBRIS,NaN,0.001403,7076.234143,...,-0.982538,0.186060,-0.587246,0.809409,-0.002133,0.999998,0.989386,-0.145308,0.988074,0.153979
4,304.0,2001.0,0.3690,0.000026,6995.466922,0.000732,PAYLOAD,0.3010,0.000039,6989.357563,...,0.990444,0.137912,0.990958,0.134172,-0.000250,1.000000,0.990765,-0.135589,0.990740,-0.135774


# data cleaning